# 03 — Radar Signal Processing

> **참고:** Udacity SFND_Radar_Target_Generation_and_Detection
> **언어:** Python (NumPy), MATLAB
> **핵심:** FMCW 신호 생성 → Range-Doppler FFT → CFAR 검출

---

## 3-1. FMCW Radar 원리

**FMCW(Frequency Modulated Continuous Wave):** 주파수를 선형적으로 변조한 연속파를 송신.



```
송신 신호 주파수
         ▲
  f_max  │     /\      /\      /\
         │    /  \    /  \    /  \
  f_min  │   /    \  /    \  /    \
         └──────────────────────────→ 시간

각 선형 상승 구간 = 1 chirp
```




**Beat Frequency 원리:**


```
송신(TX) chirp 기울기 S = (f_max - f_min) / T_chirp

타깃 거리 R에서 반사된 신호는 시간 τ = 2R/c 지연

Beat frequency  f_b = S × τ = S × 2R/c

→ 거리  R = f_b × c / (2S)
→ 속도  v = f_doppler × c / (2 × f_c)     (f_c = carrier freq)
```




---

## 3-2. 레이더 파라미터 설계



In [ ]:
import numpy as np

# 77 GHz 자동차 레이더 (TI AWR1843 기준)
fc        = 77e9          # carrier frequency [Hz]
c         = 3e8           # 빛의 속도
range_max = 200           # 최대 탐지 거리 [m]
range_res = 1.0           # 거리 해상도 [m]
vel_max   = 70 / 3.6      # 최대 속도 [m/s] (70 km/h)

# 파라미터 계산
B = c / (2 * range_res)          # 대역폭 [Hz]  → 150 MHz
T_chirp = 5.5 * 2 * range_max / c  # chirp 시간 (5.5× 이상 여유)
S = B / T_chirp                  # slope [Hz/s]
lambda_ = c / fc                 # 파장 [m]

# Range resolution: δR = c / (2B)
# Velocity resolution: δv = lambda / (2 * N_chirps * T_chirp)
print(f"Bandwidth   : {B/1e6:.1f} MHz")
print(f"Chirp time  : {T_chirp*1e6:.2f} µs")
print(f"Slope       : {S/1e12:.3f} THz/s")




---

## 3-3. IF 신호 생성 (시뮬레이션)



In [ ]:
N_chirps = 128      # 도플러 처리용 chirp 수
N_samples = 1024    # chirp 당 ADC 샘플 수
fs = 20e6           # 샘플링 주파수 [Hz]
dt = 1 / fs

# 타깃 설정
target_range = 110.0   # [m]
target_vel   = -20.0   # [m/s] (접근)

# Range-Doppler 맵 버퍼
rd_map = np.zeros((N_chirps, N_samples), dtype=complex)

for chirp_idx in range(N_chirps):
    t_chirp_start = chirp_idx * T_chirp
    for sample_idx in range(N_samples):
        t = t_chirp_start + sample_idx * dt
        # 현재 위치 (등속 운동)
        R_t = target_range + target_vel * t
        tau = 2 * R_t / c

        # TX signal phase
        phi_tx = 2 * np.pi * (fc * t + 0.5 * S * t**2)
        # RX signal phase (지연 τ)
        t_rx = t - tau
        phi_rx = 2 * np.pi * (fc * t_rx + 0.5 * S * t_rx**2)
        # Beat signal (믹서 출력)
        rd_map[chirp_idx, sample_idx] = np.exp(1j * (phi_tx - phi_rx))




---

## 3-4. Range-Doppler 맵 생성 (2D FFT)



In [ ]:
# 1D FFT along fast-time (range)
range_fft = np.fft.fft(rd_map, axis=1)
range_fft = range_fft[:, :N_samples // 2]   # 양의 주파수만

# 2D FFT along slow-time (Doppler)
rd_fft = np.fft.fftshift(
    np.fft.fft2(rd_map, axes=(0, 1)),
    axes=0
)
rd_power = 20 * np.log10(np.abs(rd_fft) + 1e-9)   # dB

# 축 정의
range_axis  = np.arange(N_samples // 2) * range_res
doppler_axis = np.linspace(-vel_max, vel_max, N_chirps)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.imshow(rd_power, aspect='auto',
           extent=[range_axis[0], range_axis[-1],
                   doppler_axis[-1], doppler_axis[0]],
           cmap='jet')
plt.colorbar(label='Power [dB]')
plt.xlabel('Range [m]')
plt.ylabel('Velocity [m/s]')
plt.title('Range-Doppler Map')
plt.show()




---

## 3-5. CFAR 검출 (CA-CFAR)

**Cell-Averaging CFAR:** 검출 셀(CUT) 주변 훈련 셀 평균 → 임계값 설정.



```
┌──────────────────────────────────────┐
│  Guard  │  Guard  │    CUT    │  Guard  │  Guard  │
│  Cells  │  Cells  │           │  Cells  │  Cells  │
├─────────┤         │           │         ├─────────┤
│Training │         │           │         │Training │
│ Cells   │         │           │         │ Cells   │
└─────────┴─────────┴───────────┴─────────┴─────────┘
 N_train   N_guard       CUT      N_guard  N_train
```


In [ ]:
def ca_cfar_2d(rd_power: np.ndarray, n_train=10, n_guard=4, pfa=1e-6):
    """
    2D CA-CFAR 검출기
    rd_power : (N_doppler, N_range) dB 파워 맵
    n_train  : 훈련 셀 수 (각 방향)
    n_guard  : 가드 셀 수 (각 방향)
    pfa      : 오경보 확률 (CFAR 임계값 조절)
    """
    N_dop, N_rng = rd_power.shape
    detection_map = np.zeros_like(rd_power)
    offset = n_train + n_guard

    # CFAR 임계값 오프셋 (지수 분포 가정)
    N_cells = (2*n_train + 2*n_guard + 1)**2 - (2*n_guard + 1)**2
    alpha = N_cells * (pfa**(-1/N_cells) - 1)   # scale factor

    for i in range(offset, N_dop - offset):
        for j in range(offset, N_rng - offset):
            # 훈련 셀 윈도우 추출 (guard+training)
            window = rd_power[i-offset:i+offset+1, j-offset:j+offset+1]
            # guard 셀 마스크
            guard  = rd_power[i-n_guard:i+n_guard+1, j-n_guard:j+n_guard+1]
            noise_sum = window.sum() - guard.sum()
            noise_avg = noise_sum / N_cells
            threshold = noise_avg + 10*np.log10(alpha)
            if rd_power[i, j] > threshold:
                detection_map[i, j] = 1
    return detection_map

detections = ca_cfar_2d(rd_power, n_train=10, n_guard=4, pfa=1e-6)
det_idx = np.where(detections == 1)
for r_idx, d_idx in zip(det_idx[1], det_idx[0]):
    r = range_axis[r_idx]
    v = doppler_axis[d_idx]
    print(f"Detection: Range={r:.1f}m, Velocity={v:.1f}m/s")




---

## 3-6. 각도 추정 (AoA)

가상 어레이(Virtual Aperture = N_tx × N_rx)로 각도 추정:



In [ ]:
# N_rx 채널 신호 (거리-도플러 후 특정 셀)
# 스티어링 벡터: a(θ) = [1, e^{j2π d sinθ/λ}, ..., e^{j2π(N-1)d sinθ/λ}]
# MUSIC 알고리즘으로 AoA 추정

def music_aoa(signal_matrix: np.ndarray, n_targets=1, d_over_lambda=0.5):
    """
    signal_matrix : (N_rx, N_snapshots) 복소 신호
    """
    N = signal_matrix.shape[0]
    # 공분산 행렬
    R = signal_matrix @ signal_matrix.conj().T / signal_matrix.shape[1]
    # 고유분해
    eigenvalues, eigenvectors = np.linalg.eigh(R)
    # 잡음 부분공간 (작은 고유값)
    noise_subspace = eigenvectors[:, :-n_targets]

    # 각도 스캔
    angles = np.linspace(-90, 90, 181)
    spectrum = []
    for theta in angles:
        a = np.exp(1j * 2*np.pi * d_over_lambda *
                   np.arange(N) * np.sin(np.deg2rad(theta)))
        a = a.reshape(-1, 1)
        P = 1.0 / (a.conj().T @ noise_subspace @ noise_subspace.conj().T @ a)
        spectrum.append(abs(P[0, 0]))
    return angles, np.array(spectrum)




---

## 3-7. MATLAB 코드 (Udacity 과제 스타일)



```matlab
%% FMCW 파라미터
fc = 77e9; c = 3e8;
range_max = 200; range_res = 1;
vel_max = 70/3.6;

B = c / (2*range_res);
T_chirp = 5.5 * 2 * range_max / c;
S = B / T_chirp;

%% 타깃
target_R = 110; target_v = -20;

%% IF 신호 생성
Nd = 128; Nr = 1024;
t = linspace(0, Nd*T_chirp, Nr*Nd);
Mix = zeros(1, length(t));

for i = 1:length(t)
    t_i = t(i);
    R_i = target_R + target_v * t_i;
    tau = 2*R_i/c;
    Tx = cos(2*pi*(fc*t_i + S*t_i^2/2));
    Rx = cos(2*pi*(fc*(t_i-tau) + S*(t_i-tau)^2/2));
    Mix(i) = Tx * Rx;
end

Mix = reshape(Mix, [Nr, Nd]);

%% Range FFT
sig_fft1 = fft(Mix, Nr);
sig_fft1 = abs(sig_fft1(1:Nr/2));

%% 2D FFT (Range-Doppler)
sig_fft2 = fft2(Mix, Nr, Nd);
sig_fft2 = fftshift(sig_fft2, 2);
RDM = abs(sig_fft2);
RDM = 10*log10(RDM);

%% CFAR
Tr=10; Td=8; Gr=4; Gd=4; offset=6;
noise_level = zeros(Nr, Nd);
for i = Tr+Gr+1:(Nr-(Gr+Tr))
    for j = Td+Gd+1:(Nd-(Gd+Td))
        noise_level(i,j) = sum(db2pow(RDM(i-Tr-Gr:i+Tr+Gr, j-Td-Gd:j+Td+Gd)),'all');
        noise_level(i,j) = noise_level(i,j) - sum(db2pow(RDM(i-Gr:i+Gr, j-Gd:j+Gd)),'all');
    end
end
```




---

## 3-8. 핵심 개념 정리

| 개념 | 핵심 |
|------|------|
| Beat frequency | Δf = S × 2R/c, 거리↔주파수 선형 매핑 |
| Range FFT | fast-time 차원, 해상도 δR = c/2B |
| Doppler FFT | slow-time 차원, 해상도 δv = λ/2N_chirp T_chirp |
| CA-CFAR | 주변 노이즈 평균으로 임계값 적응, Pfa 제어 |
| Virtual aperture | N_tx × N_rx 가상 안테나 → 각도 해상도 향상 |
| MUSIC | 잡음 부분공간 직교성 이용 초분해능 각도 추정 |

---

## 다음 단계

→ **04_calibration.md** : Camera-LiDAR 외부 파라미터 보정 (PnP, SVD)
